In [ ]:
import numpy as np
import scipy.integrate
from perfetto.trace_processor import TraceProcessor
import pandas as pd
from typing import Dict, Any, Optional

In [ ]:
class LBO(object):
    GC_SLICE_SQL_LIKE = "% % GC"

    def __init__(self, trace_path: str):
        self.tp = TraceProcessor(trace=trace_path)
        self.__create_view_freq_counter()
        self.__create_view_thread_state_with_name()
        self.__create_view_gc_slice()
        self.__create_view_gc_slice_cpu_time()
        self.__create_virtual_table_freq("gc_slice_freq", "gc_slice_thread_state")
        self.__create_virtual_table_freq("thread_freq", "thread_state_with_name")
    
    def __query(self, q: str):
        return self.tp.query(q)
    
    def query_pandas(self, q: str) -> pd.DataFrame:
        return self.__query(q).as_pandas_dataframe()

    def query_single(self, q: str) -> Dict[str, Any]:
        return next(self.__query(q)).__dict__

    def execute(self, q: str):
        self.__query(q)

    def __create_view_thread_state_with_name(self):
        self.execute("""
            CREATE VIEW thread_state_with_name AS
            SELECT
                process.upid,
                process.name AS process_name,
                thread.utid,
                thread.name AS thread_name,
                thread_state.cpu,
                thread_state.ts,
                thread_state.dur,
                thread_state.state
            FROM thread_state
            INNER JOIN thread ON thread.utid = thread_state.utid
            INNER JOIN process ON process.upid = thread.upid
        """)

    def __create_view_freq_counter(self):
        self.execute("""
            CREATE VIEW freq_counter AS
            SELECT
                ts,
                /* use max signed int64 as the end ts for the last counter sample*/
                (LEAD(ts, 1, 9223372036854775807) OVER (PARTITION BY cpu ORDER BY ts) - ts) AS dur,
                cpu,
                value AS freq_khz
            FROM counter
            INNER JOIN cpu_counter_track ON counter.track_id=cpu_counter_track.id
            WHERE cpu_counter_track.name='cpufreq'
        """)

    def __create_view_gc_slice(self):
        self.execute("""
            CREATE VIEW gc_slice AS
            SELECT
                slice.id,
                slice.ts,
                slice.dur,
                slice.name,
                thread_track.utid
            FROM slice
            INNER JOIN thread_track ON thread_track.id = slice.track_id
            WHERE slice.name LIKE '{}'
        """.format(LBO.GC_SLICE_SQL_LIKE))

    def __create_view_gc_slice_cpu_time(self):
        self.execute("""
            CREATE VIRTUAL TABLE gc_slice_thread_state
            USING SPAN_JOIN(gc_slice PARTITIONED utid, thread_state_with_name PARTITIONED utid)
        """)

    def get_gc_slices(self) -> pd.DataFrame:
        # Enable scheduling and atrace_categories: "dalvik"
        return self.query_pandas("""
            SELECT *
            FROM gc_slice
        """)

    def get_gc_slices_cpu_time(self, process: Optional[str] = None) -> pd.DataFrame:
        # Enable scheduling and atrace_categories: "dalvik"
        return self.query_pandas("""
            SELECT
                id,
                name,
                upid,
                process_name,
                utid,
                thread_name,
                SUM(dur) AS cpu_time
            FROM gc_slice_thread_state
            WHERE state = 'Running' {}
            GROUP BY id, cpu
        """.format("AND process_name = '{}'".format(process) if process else ""))

    def get_process_cpu_time(self, process: Optional[str] = None) -> pd.DataFrame:
        return self.query_pandas("""
            SELECT
                upid,
                process_name,
                cpu,
                SUM(dur) AS cpu_time
            FROM thread_state_with_name
            WHERE state = 'Running' {}
            GROUP BY upid, cpu
        """.format("AND process_name = '{}'".format(process)))

    def __create_virtual_table_freq(self, table_name: str, base_table: str):
        self.execute(f"""
            CREATE VIRTUAL TABLE {table_name}
            USING SPAN_JOIN({base_table} PARTITIONED cpu, freq_counter PARTITIONED cpu)
        """)

    def get_gc_slices_cycles(self, process: str) -> pd.DataFrame:
        return self.query_pandas(f"""
            SELECT
                id,
                name,
                upid,
                process_name,
                utid,
                thread_name,
                cpu,
                SUM(freq_khz * dur / 1000000) AS cycles
            FROM gc_slice_freq
            WHERE process_name = '{process}' AND state = 'Running'
            GROUP BY id, cpu
        """)

    def get_process_cycles(self, process: str) -> pd.DataFrame:
        return self.query_pandas(f"""
            SELECT
                upid,
                process_name,
                utid,
                thread_name,
                cpu,
                SUM(freq_khz * dur / 1000000) AS cycles
            FROM thread_freq
            WHERE process_name = '{process}' AND state = 'Running'
            GROUP BY upid, cpu
        """)

    def distillation(self, process: str) -> pd.DataFrame:
        gc_df = self.get_gc_slices_cycles(process)[["cpu", "cycles"]].groupby("cpu").sum()
        process_df = self.get_process_cycles(process)[["cpu", "cycles"]].groupby("cpu").sum()
        df = pd.concat(gc_df.align(process_df, fill_value = 0), axis='columns')
        df.columns = ["distilled", "total"]
        return df


    def cpufreq(self) -> pd.DataFrame:
        return self.query_pandas("""
            SELECT * FROM freq_counter
        """)

    def integrate_mem(self, process: str, mem_type: str = "mem.rss.anon") -> int:
        # Enable ftrace_events: "kmem/rss_stat" in Perfetto
        # mem_type can be "mem.rss.anon", "mem.rss.file", "mem.rss.shmem", "mem.swap"
        df = self.tp.query("""
            SELECT ts, value, process_counter_track.name, process.name
            FROM counter
            INNER JOIN process_counter_track ON process_counter_track.id = counter.track_id
            INNER JOIN process ON process.upid = process_counter_track.upid
            WHERE process_counter_track.name = '{}' AND process.name = '{}'
            ORDER BY ts ASC
        """.format(mem_type, process)).as_pandas_dataframe()
        print(df)
        # value is in byte
        # ts is in nanosecond
        return scipy.integrate.trapezoid(df["value"], df["ts"]) / 1e9 / 1024 / 1024 # MB * second

In [ ]:
lbo = LBO("example-SS.perfetto-trace")

In [ ]:
lbo.get_gc_slices()

In [ ]:
lbo.get_process_cpu_time()

In [ ]:
lbo.get_gc_slices_cpu_time("com.google.android.apps.maps")

In [ ]:
lbo.get_process_cpu_time("com.google.android.apps.maps")

In [ ]:
lbo.distillation("com.google.android.apps.maps")